<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDS0321ENSkillsNetwork26802033-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Hands-on Lab: Interactive Visual Analytics with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [1]:
try:
    import piplite
except ImportError:
    piplite = None

if piplite is not None:
    await piplite.install(['folium', 'pandas'])


In [2]:
import folium
import pandas as pd
import requests


In [3]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)


In [4]:
## Task 1: Mark all launch sites on a map
# define the world map
world_map = folium.Map()

# display world map
world_map



First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [5]:
# Download and read the `spacex_launch_geo.csv`
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'

try:
    from js import fetch
except ImportError:
    fetch = None

if fetch is not None:
    resp = await fetch(URL)
    spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
    spacex_df = pd.read_csv(spacex_csv_file)
else:
    response = requests.get(URL, timeout=30)
    response.raise_for_status()
    spacex_df = pd.read_csv(io.BytesIO(response.content))


Now, you can take a look at what are the coordinates for each site.


In [6]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [7]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [8]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


Completed below: each launch site is shown with a 1 km circle, popup, and label.


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [9]:
# Initialize the map and add a circle and label for each launch site
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
sites = folium.FeatureGroup(name='Launch Sites')

for lat, lng, label in zip(
    launch_sites_df['Lat'], launch_sites_df['Long'], launch_sites_df['Launch Site']
):
    coordinate = [lat, lng]
    folium.Circle(
        coordinate,
        radius=1000,
        color='#d35400',
        fill=True,
        fill_opacity=0.4,
        popup=folium.Popup(label),
    ).add_to(sites)
    folium.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(150, 25),
            icon_anchor=(0, 0),
            html=f'<div style="font-size: 12px; color: #d35400;"><b>{label}</b></div>',
        ),
    ).add_to(sites)

site_map.add_child(sites)
site_map


The generated map with marked launch sites should look similar to the following:


The launch sites are not on the Equator: three are near 28.6° N in Florida and VAFB is near 34.6° N in California. All four sites are close to a coastline, which is useful for launch safety and over-water trajectories.

Explore the map to compare the sites' proximity to the Equator and coast.


In [10]:
# Launch outcome markers are added below after marker colors are assigned.


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [11]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [12]:
marker_cluster = MarkerCluster()


Completed below: `marker_color` is green for successful launches and red for failures.


In [13]:
# Assign green markers to successful launches and red markers to failed launches
spacex_df['marker_color'] = spacex_df['class'].map({1: 'green', 0: 'red'})
spacex_df[['Launch Site', 'class', 'marker_color']].tail(10)


,Launch Site,class,marker_color
46,KSC LC-39A,1,green
47,KSC LC-39A,1,green
48,KSC LC-39A,1,green
49,CCAFS SLC-40,1,green
50,CCAFS SLC-40,1,green
51,CCAFS SLC-40,0,red
52,CCAFS SLC-40,0,red
53,CCAFS SLC-40,0,red
54,CCAFS SLC-40,1,green
55,CCAFS SLC-40,0,red


Completed below: each launch is shown as a color-coded marker in a `MarkerCluster`.


In [14]:
# Add one marker for each launch outcome to the marker cluster
site_map.add_child(marker_cluster)

for _, record in spacex_df.iterrows():
    outcome_text = 'Success' if record['class'] == 1 else 'Failure'
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        popup=f"<b>Site:</b> {record['Launch Site']}<br><b>Outcome:</b> {outcome_text}",
        icon=folium.Icon(
            color='white',
            icon_color=record['marker_color'],
            icon='info-sign',
        ),
    )
    marker_cluster.add_child(marker)

site_map


Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


In this dataset, KSC LC-39A has the highest observed launch success rate (10/13, about 76.9%), followed by CCAFS SLC-40 (3/7, 42.9%), VAFB SLC-4E (4/10, 40.0%), and CCAFS LC-40 (7/26, 26.9%). These are descriptive rates for this dataset and do not establish that location caused the differences.


In [15]:
# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [16]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


In [17]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

Completed below for CCAFS LC-40: a nearby coastline point was selected and its straight-line distance was calculated.


In [18]:
# Use the CCAFS LC-40 launch site and a nearby coastline point identified on the map
launch_site = launch_sites_df.loc[
    launch_sites_df['Launch Site'] == 'CCAFS LC-40'
].iloc[0]
launch_site_lat = float(launch_site['Lat'])
launch_site_lon = float(launch_site['Long'])
coastline_lat = 28.56367
coastline_lon = -80.57163

distance_coastline = calculate_distance(
    launch_site_lat, launch_site_lon, coastline_lat, coastline_lon
)
print(f'Distance from CCAFS LC-40 to the selected coastline point: {distance_coastline:.2f} km')


Distance from CCAFS LC-40 to the selected coastline point: 0.58 km


In [19]:
# Mark the selected coastline point and display its distance from the launch site
coastline_coordinate = [coastline_lat, coastline_lon]
distance_marker = folium.Marker(
    coastline_coordinate,
    icon=DivIcon(
        icon_size=(150, 25),
        icon_anchor=(0, 0),
        html=f'<div style="font-size: 12px; color: #d35400;"><b>{distance_coastline:.2f} km</b></div>',
    ),
    popup='Selected coastline point',
)
site_map.add_child(distance_marker)
site_map


Completed below: the CCAFS LC-40 site and selected coastline point are connected by a `PolyLine`.


In [20]:
# Draw and display a line between the launch site and coastline point
coordinates = [
    [launch_site_lat, launch_site_lon],
    coastline_coordinate,
]
lines = folium.PolyLine(
    locations=coordinates,
    weight=2,
    color='blue',
    tooltip=f'{distance_coastline:.2f} km to coastline',
)
site_map.add_child(lines)
site_map


Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


Completed below with a Cape Canaveral city-center example. Railway and highway proximity is discussed at the end; the launch CSV does not contain road or rail coordinates.


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [21]:
# Add a nearby city-center reference point and measure its straight-line distance
# Cape Canaveral city-center coordinates (OpenStreetMap/Nominatim)
city_label = 'Cape Canaveral city center'
city_lat = 28.4513556
city_lon = -80.5283059
city_coordinate = [city_lat, city_lon]
distance_city = calculate_distance(
    launch_site_lat, launch_site_lon, city_lat, city_lon
)

city_marker = folium.Marker(
    city_coordinate,
    popup=f'{city_label}: {distance_city:.2f} km from CCAFS LC-40 (straight-line)',
    icon=DivIcon(
        icon_size=(180, 25),
        icon_anchor=(0, 0),
        html=f'<div style="font-size: 12px; color: #1f618d;"><b>{city_label}</b></div>',
    ),
)
city_marker.add_to(site_map)
city_line = folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon], city_coordinate],
    weight=2,
    color='orange',
    tooltip=f'{distance_city:.2f} km to {city_label}',
)
site_map.add_child(city_line)


In [22]:
# Summarize the two measured proximity examples
proximity_distances = pd.DataFrame(
    [
        {'Feature': 'Selected coastline point', 'Distance (km)': distance_coastline},
        {'Feature': city_label, 'Distance (km)': distance_city},
    ]
)
proximity_distances


,Feature,Distance (km)
0,Selected coastline point,0.579758
1,Cape Canaveral city center,13.239166


In [23]:
# Display the final map with launch outcomes and proximity lines
site_map


### Proximity findings


- **Coastline:** Yes. The launch sites are coastal. For the selected CCAFS LC-40 point, the measured straight-line distance to the chosen coastline point is about 0.58 km.
- **Cities:** Launch pads are on controlled coastal ranges rather than in dense city centers. The selected Cape Canaveral city-center point is about 13.24 km from CCAFS LC-40 in a straight line. The distance varies by site and by whether one measures to a city center or boundary.

These are geographic observations, not proof that proximity caused launch outcomes. The coastline and city figures are straight-line distances; railway and highway distances require specific feature coordinates selected from the map.
The launch sites are far away from railways and aminities and close the coast line as this reduces the risks in case a launch fails and so the launch modules don't creat damage when discarded.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Pratiksha Verma](https://www.linkedin.com/in/pratiksha-verma-6487561b1/)


<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>


### <h3 align="center"> IBM Corporation 2022. All rights reserved. <h3/>
